# Segmentação de arvores e análise

## Preparação dos dados para segmentação:

Instalação das bibliotecas neessárias:

In [2]:
!pip install rasterio

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Importação das funções e pacotes

In [4]:
from skimage.segmentation import felzenszwalb, slic, quickshift, watershed
from skimage.segmentation import mark_boundaries
from skimage.util import img_as_float
from skimage.filters import sobel
from skimage.color import rgb2gray
from skimage import color
import rasterio
from matplotlib import pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
from rasterio.plot import show

Abrir a imagem com o rasterio, convertendo num numpy array:

In [5]:
path_img = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_RGB_NEW.tif'
src_rgb = rasterio.open(path_img)
im = src_rgb.read()
im  = im.transpose([1,2,0])
'''R = src_rgb.read(3)
G = src_rgb.read(2)
B = src_rgb.read(1)


R = R / 10000
G = G / 10000
B = B / 10000

R = R * 255
G = G * 255
B = B * 255

R = R.astype('uint8')
G = G.astype('uint8')
B = B.astype('uint8')

im = np.dstack((R,G,B))'''

"R = src_rgb.read(3)\nG = src_rgb.read(2)\nB = src_rgb.read(1)\n\n\nR = R / 10000\nG = G / 10000\nB = B / 10000\n\nR = R * 255\nG = G * 255\nB = B * 255\n\nR = R.astype('uint8')\nG = G.astype('uint8')\nB = B.astype('uint8')\n\nim = np.dstack((R,G,B))"

Plotando a imagem com o matplotlib:

In [6]:
plt.figure(figsize=[12,12])
plt.imshow(im)
plt.axis('off')

(np.float64(-0.5), np.float64(6941.5), np.float64(7293.5), np.float64(-0.5))

<Figure size 1200x1200 with 1 Axes>

Abrir o poligono da área de interesse:

In [7]:
AOI = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_new.shp')

## Segmentação:

Aplicação do quickshift (Ver artigos relacionados)

In [ ]:
segments_quick2 = quickshift(im, kernel_size=3, max_dist=5, ratio=0.04)

In [ ]:
plt.figure(figsize=[16,16])
plt.imshow(mark_boundaries(im, segments_quick2))
plt.axis('off')

(np.float64(-0.5), np.float64(6941.5), np.float64(7293.5), np.float64(-0.5))

<Figure size 1600x1600 with 1 Axes>

In [ ]:
segments_quick2 = segments_quick2.astype('uint16')

Conversão para poligono e extração das caracteristicas espectrais:

In [ ]:
from rasterio import features
from shapely.geometry import shape
import geopandas as gpd
import shapely
import json
from rasterio.mask import mask
import pandas as pd

Primeiro, converter os segmentos em poligonos usando geopandas e rasterio:

In [ ]:
find_shapes = features.shapes(segments_quick2, transform=src_rgb.transform)

In [ ]:
polygons = [shapely.geometry.Polygon(shape[0]["coordinates"][0]) for shape in find_shapes]

In [ ]:
crs = src_rgb.crs
Poly_gdf = gpd.GeoDataFrame(crs=crs, geometry=polygons)

Salvar os poligonos em um JSON:

In [ ]:
Poly_gdf.to_file("polygons.json")

In [ ]:
Poly_gdf.boundary.plot()

<Axes: xlabel='Easting [metre]', ylabel='Northing [metre]'>

<Figure size 640x480 with 1 Axes>

Obter somente os poligonos da área de interesse:

In [ ]:
clip_poly = Poly_gdf.clip(AOI)

In [ ]:
clip_poly.boundary.plot()

<Axes: xlabel='Easting [metre]', ylabel='Northing [metre]'>

<Figure size 640x480 with 1 Axes>

Extrair a média espectral de cada segmento:

In [ ]:
clip_poly = clip_poly.reset_index(drop=True)

In [ ]:
json_dataset = json.loads(clip_poly.to_json())['features']

In [ ]:
path_img_mult = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_NEW_MULTI.tif'
src = rasterio.open(path_img_mult)

X = []
Y = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=0)

  out_image = out_image.transpose([1,2,0])

  R_mean = out_image[:,:,0][np.nonzero(out_image[:,:,0])].mean()
  G_mean = out_image[:,:,1][np.nonzero(out_image[:,:,1])].mean()
  B_mean = out_image[:,:,2][np.nonzero(out_image[:,:,2])].mean()
  N_mean = out_image[:,:,3][np.nonzero(out_image[:,:,3])].mean()
  R_std = out_image[:,:,0][np.nonzero(out_image[:,:,0])].std()
  G_std = out_image[:,:,1][np.nonzero(out_image[:,:,1])].std()
  B_std = out_image[:,:,2][np.nonzero(out_image[:,:,2])].std()
  N_std = out_image[:,:,3][np.nonzero(out_image[:,:,3])].std()




  X.append([R_mean,G_mean,B_mean,N_mean,R_std,G_std,B_std,N_std])

A saída de streaming foi truncada nas últimas 5000 linhas.
/tmp/ipykernel_2935/2157018459.py:15: RuntimeWarning: Mean of empty slice.
  N_mean = out_image[:,:,3][np.nonzero(out_image[:,:,3])].mean()
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_2935/2157018459.py:12: RuntimeWarning: Mean of empty slice.
  R_mean = out_image[:,:,0][np.nonzero(out_image[:,:,0])].mean()
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dt

In [ ]:
X = np.array(X)

## Aplicação do K-Means

Geramos o dataframe com a informação das médias espectrais por segmentos:

In [ ]:
df = pd.DataFrame(X, columns=['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std'])

In [ ]:
df

            R_Mean       G_Mean       B_Mean       N_Mean       R_Std  \
0              NaN          NaN          NaN          NaN         NaN   
1              NaN          NaN          NaN          NaN         NaN   
2              NaN          NaN          NaN          NaN         NaN   
3              NaN          NaN          NaN          NaN         NaN   
4              NaN          NaN          NaN          NaN         NaN   
...            ...          ...          ...          ...         ...   
166367         NaN          NaN          NaN          NaN         NaN   
166368         NaN          NaN          NaN          NaN         NaN   
166369  408.111111   452.111111   406.111111  1339.277778   51.987059   
166370         NaN          NaN          NaN          NaN         NaN   
166371  969.333333  1212.666667  1390.333333  1662.166667  173.764847   

             G_Std       B_Std       N_Std  
0              NaN         NaN         NaN  
1              NaN         NaN   

In [ ]:
df.dropna(inplace=True)

Aplicamos o Kmeans:

In [ ]:
from sklearn.cluster import KMeans

# Apply K-Means clustering
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10) # You can change the number of clusters
kmeans_clusters = kmeans.fit_predict(df[['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std']])

# Add K-Means cluster labels to the DataFrame
df['kmeans_cluster'] = kmeans_clusters

display(df.head())

         R_Mean      G_Mean      B_Mean       N_Mean       R_Std       G_Std  \
289  622.059459  734.670270  669.221622  1946.318919   93.138674  102.884477   
290  575.494382  661.382022  592.314607  1560.337079   82.741390  102.830668   
291  627.989362  753.606383  730.058511  1804.404255  117.009228  138.669224   
292  648.189274  753.381703  769.671924  1222.548896   79.557781   92.890812   
297  612.924528  736.150943  756.716981  1632.698113   42.417889   49.736436   

          B_Std       N_Std  kmeans_cluster  
289  113.241381  258.600748               3  
290   90.490746  366.842340               1  
291  148.149981  379.788626               3  
292  128.273268  279.971712               4  
297   55.419286  171.135341               1  

Aplicamos o resultado do Kmeans para os segmentos e visualizamos:

In [ ]:
clip_poly['kmeans_cluster'] = df['kmeans_cluster']

In [ ]:
import matplotlib.pyplot as plt

# Plot the GeoDataFrame, colored by cluster
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
clip_poly.plot(column='kmeans_cluster', ax=ax, legend=True, cmap='magma') # Changed colormap to 'viridis'
plt.title('Polygons Colored by kmeans_cluster')
plt.axis('off')
plt.show()

<Figure size 1500x1500 with 2 Axes>

Salvamos o resultado em um JSON:

In [ ]:
Poly_gdf.to_file('polygons.json')

In [ ]:
clip_poly[clip_poly['kmeans_cluster'] == 3].to_file('polygons_arvores.json')

Selecionamos somente o cluster de arvores:

In [ ]:
import rasterio.plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
clip_poly[clip_poly['kmeans_cluster'] == 3].boundary.plot(ax=ax, color='red')
plt.title('Original Image')
plt.axis('off')
plt.show()

<Figure size 1500x1500 with 1 Axes>

In [ ]:
path_grama = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Grama.gpkg'
path_arvore = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Arvores.gpkg'

In [ ]:
gdf_grama = gpd.read_file(path_grama)
gdf_arvore = gpd.read_file(path_arvore)

In [ ]:
gdf_grama['Classe'] = 'grama'
gdf_arvore['Classe'] = 'arvore'

In [ ]:
gdf_full = pd.concat([gdf_grama, gdf_arvore])
gdf_full.to_crs(clip_poly.crs, inplace=True)

In [ ]:
gdf_arvore_seg = clip_poly[clip_poly['kmeans_cluster'] == 3]

In [ ]:
gdf_arvore_seg_filter = gdf_arvore_seg.sjoin(gdf_full, how='inner', predicate='intersects')

In [ ]:
gdf_arvore_seg_filter.plot()

<Axes: xlabel='Easting [metre]', ylabel='Northing [metre]'>

<Figure size 640x480 with 1 Axes>

In [ ]:
gdf_arvore_seg_filter

                                                 geometry  kmeans_cluster  \
689     POLYGON ((483321.98 6772197.27, 483321.98 6772...             3.0   
698     POLYGON ((483299.48 6772202.77, 483299.48 6772...             3.0   
2558    POLYGON ((482636.48 6772249.77, 482636.48 6772...             3.0   
3380    POLYGON ((482825.48 6772510.77, 482825.48 6772...             3.0   
3381    POLYGON ((482811.48 6772508.77, 482811.48 6772...             3.0   
...                                                   ...             ...   
156114  POLYGON ((483703.98 6775462.27, 483703.98 6775...             3.0   
158805  POLYGON ((484651.48 6775098.77, 484651.48 6775...             3.0   
163537  POLYGON ((484834.98 6775141.27, 484834.98 6775...             3.0   
163583  POLYGON ((484834.98 6775187.77, 484834.98 6775...             3.0   
163732  POLYGON ((484850.48 6775183.27, 484850.48 6775...             3.0   

        index_right  Classe  
689              63   grama  
698            

In [ ]:
src = rasterio.open(path_img_mult)

In [ ]:
img = src.read()

In [ ]:
img.shape

(4, 7294, 6942)

In [ ]:
nir = img[3]
red = img[0]

In [ ]:
nir.shape

(7294, 6942)

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(nir, cmap='gray')
plt.title('NIR Band')
plt.axis('off')
plt.show()

<Figure size 1000x1000 with 1 Axes>

In [ ]:
import numpy as np

# Calculate NDVI
# Avoid division by zero by adding a small epsilon to the denominator
ndvi = (nir.astype(float) - red.astype(float)) / (nir.astype(float) + red.astype(float) + 1e-8)

# Plot NDVI
plt.figure(figsize=(10, 10))
plt.imshow(ndvi, cmap='RdYlGn', vmin=0, vmax=1) # RdYlGn is a common colormap for NDVI
plt.colorbar(label='NDVI Value')
plt.title('Normalized Difference Vegetation Index (NDVI)')
plt.axis('off')
plt.show()

<Figure size 1000x1000 with 2 Axes>

In [ ]:
nir.max()

np.uint16(8594)

In [ ]:
from skimage.feature import canny
from skimage.util import img_as_ubyte

# Ensure nir is in the correct format (uint8) for Canny
nir_uint = nir/10000

# Apply Canny edge detection
edges = canny(nir_uint, sigma=1)

# Plot the edges
plt.figure(figsize=(10, 10))
plt.imshow(edges, cmap='gray')
plt.title('Canny Edges of NIR Band')
plt.axis('off')
plt.show()

<Figure size 1000x1000 with 1 Axes>

In [ ]:
img_rgb = img[0:3]

In [ ]:
img_rgb = img_rgb.transpose([1,2,0])

In [ ]:
from skimage.color import rgb2hsv

# Convert RGB to HSV
img_hsv = rgb2hsv(img_rgb)

# Extract the intensity (Value) channel, which is the third channel (index 2)
intensity = img_hsv[:, :, 2]

# Display the intensity image
plt.figure(figsize=(10, 10))
plt.imshow(intensity, cmap='gray')
plt.title('Intensity Channel (HSV)')
plt.axis('off')
plt.show()

<Figure size 1000x1000 with 1 Axes>

In [ ]:
# Convert edges (boolean) to float for stacking
edges_float = edges.astype(np.float32)

# Stack intensity and edges as two separate bands
# The order here is intensity first, then edges
stacked_bands = np.stack([intensity, edges_float, ndvi], axis=0)

# Get metadata from the original source (src) for writing the new TIFF
profile = src.profile

# Update profile for the new stacked image
# Change the number of bands to 2 and data type to float32
profile.update(
    dtype=rasterio.float32,
    count=3,
    nodata=None
)

# Define the output path for the new TIFF file
output_filepath = 'features_stack.tif'

# Write the stacked image to a new TIFF file
with rasterio.open(output_filepath, 'w', **profile) as dst:
    dst.write(stacked_bands)


In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg_filter.to_json())['features']

X = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)
  classe_values = json_dataset[i]['properties']['Classe']

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X.append([mean_intensity, mean_edges, mean_ndvi,classe_values])


In [ ]:
df = pd.DataFrame(X, columns=['mean_intensity','mean_edges','mean_ndvi','classe'])

In [ ]:
import seaborn as sns

# Create a pairplot with 'classe' as hue
sns.pairplot(df, hue='classe')
plt.show()

<Figure size 845.375x750 with 12 Axes>

In [ ]:
df

     mean_intensity  mean_edges  mean_ndvi  classe
0          0.011528    0.174528   0.639673   grama
1          0.011252    0.055843   0.648141   grama
2          0.009893    0.170626   0.615178  arvore
3          0.010910    0.136578   0.554167  arvore
4          0.010956    0.115171   0.593257  arvore
..              ...         ...        ...     ...
453        0.010180    0.035971   0.762713   grama
454        0.013191    0.000000   0.483285   grama
455        0.012480    0.000000   0.617056   grama
456        0.011864    0.000000   0.637704   grama
457        0.011233    0.034014   0.620410   grama

[458 rows x 4 columns]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Separate features (X) and target (y)
X = df.drop('classe', axis=1) # Features are all columns except 'classe'
y = df['classe'] # Target is the 'classe' column

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Classifier
# You can adjust parameters like n_estimators (number of trees) as needed
rf_classifier = RandomForestClassifier(n_estimators=500, random_state=42)

# Train the model
rf_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test)

# Evaluate the model
print("Random Forest Classifier Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}")

Random Forest Classifier Report:
              precision    recall  f1-score   support

      arvore       0.80      0.92      0.85        38
       grama       0.94      0.83      0.88        54

    accuracy                           0.87        92
   macro avg       0.87      0.88      0.87        92
weighted avg       0.88      0.87      0.87        92

Accuracy Score: 0.87


In [ ]:
# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=rf_classifier.classes_)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_classifier.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

<Figure size 640x480 with 2 Axes>

In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg.to_json())['features']

X_pred = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X_pred.append([mean_intensity, mean_edges, mean_ndvi])

In [ ]:
df_pred = pd.DataFrame(X_pred, columns=['mean_intensity','mean_edges','mean_ndvi'])

In [ ]:
gdf_arvore_seg['predicted_classe'] = rf_classifier.predict(df_pred)

/usr/local/lib/python3.13/dist-packages/geopandas/geodataframe.py:2011: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
gdf_arvore_seg

                                                 geometry  kmeans_cluster  \
289     POLYGON ((483237.98 6772099.77, 483237.98 6772...             3.0   
291     POLYGON ((483245.98 6772102.77, 483245.98 6772...             3.0   
522     POLYGON ((483237.48 6772124.27, 483237.48 6772...             3.0   
550     POLYGON ((483241.48 6772140.77, 483241.48 6772...             3.0   
557     POLYGON ((483243.48 6772148.27, 483243.48 6772...             3.0   
...                                                   ...             ...   
166311  POLYGON ((484861.48 6775653.77, 484861.48 6775...             3.0   
166314  POLYGON ((484893.98 6775653.27, 484893.98 6775...             3.0   
166315  POLYGON ((484894.48 6775655.27, 484894.48 6775...             3.0   
166316  POLYGON ((484903.48 6775657.77, 484903.48 6775...             3.0   
166318  POLYGON ((484898.98 6775666.77, 484900.48 6775...             3.0   

       predicted_classe  
289              arvore  
291              arvore

In [ ]:
gdf_arvore_seg['area'] = gdf_arvore_seg.area

/usr/local/lib/python3.13/dist-packages/geopandas/geodataframe.py:2011: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
tree_polys = gdf_arvore_seg.copy()

## Seleção das arvores e aplicação da Alometria:

Selecionamos os segmentos que pertence ao cluster das arvores:

In [ ]:
tree_polys

                                                 geometry  kmeans_cluster  \
289     POLYGON ((483237.98 6772099.77, 483237.98 6772...             3.0   
291     POLYGON ((483245.98 6772102.77, 483245.98 6772...             3.0   
522     POLYGON ((483237.48 6772124.27, 483237.48 6772...             3.0   
550     POLYGON ((483241.48 6772140.77, 483241.48 6772...             3.0   
557     POLYGON ((483243.48 6772148.27, 483243.48 6772...             3.0   
...                                                   ...             ...   
166311  POLYGON ((484861.48 6775653.77, 484861.48 6775...             3.0   
166314  POLYGON ((484893.98 6775653.27, 484893.98 6775...             3.0   
166315  POLYGON ((484894.48 6775655.27, 484894.48 6775...             3.0   
166316  POLYGON ((484903.48 6775657.77, 484903.48 6775...             3.0   
166318  POLYGON ((484898.98 6775666.77, 484900.48 6775...             3.0   

       predicted_classe       area  
289              arvore   46.25000  
2

Aplicamos a formula da alometria:

In [ ]:
import numpy as np

#C1
a1 = 0.5
b1 = 0.7
tree_polys['Power_tree_size_C1'] = a1 * (tree_polys['area'] ** b1)
tree_polys['Linear_tree_size_C1'] = a1 + (tree_polys['area'] * b1)


#C2
a2 = 0.6
b2 = 0.8
tree_polys['Power_tree_size_C2'] = a2 * (tree_polys['area'] ** b2)
tree_polys['Linear_tree_size_C2'] = a2 + (tree_polys['area'] * b2)


#C3
a3 = 2.5
b3 = 0.4
tree_polys['Power_tree_size_C3'] = a3 * (tree_polys['area'] ** b3)
tree_polys['Linear_tree_size_C3'] = a3 + (tree_polys['area'] * b3)

#C4
a4 = 4
b4 = 0.5
tree_polys['Power_tree_size_C4'] = a4 * (tree_polys['area'] ** b4)
tree_polys['Linear_tree_size_C4'] = a4 + (tree_polys['area'] * b4)

#C5

a5 = 0.4
b5 = 0.9
tree_polys['Power_tree_size_C5'] = a5 * (tree_polys['area'] ** b5)
tree_polys['Linear_tree_size_C5'] = a5 + (tree_polys['area'] * b5)



a6 = 1.35
b6 = 0.72
tree_polys['Power_tree_size_C6'] = a6 * (tree_polys['area'] ** b6)
tree_polys['Linear_tree_size_C6'] = a6 + (tree_polys['area'] * b6)

In [ ]:
columns = ['Power_tree_size_C1', 'Linear_tree_size_C1', 'Power_tree_size_C2', 'Linear_tree_size_C2',
           'Power_tree_size_C3', 'Linear_tree_size_C3', 'Power_tree_size_C4', 'Linear_tree_size_C4',
           'Power_tree_size_C5', 'Linear_tree_size_C5', 'Power_tree_size_C6', 'Linear_tree_size_C6']

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=tree_polys[columns])
plt.title('Box Plot of Tree Size Metrics for Different Constants')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

<Figure size 1500x800 with 1 Axes>

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=tree_polys['Power_tree_size_C3'])
plt.title('Box Plot of Tree Size Metrics for C3')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

<Figure size 1500x800 with 1 Axes>

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(tree_polys['Power_tree_size_C3'], bins=30, edgecolor='black')
plt.title('Histogram of Power_tree_size_C3')
plt.xlabel('Power_tree_size_C3')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

<Figure size 1000x600 with 1 Axes>

Obtemos somente arvores maiores que 7 metros:

In [ ]:
filter_tree = tree_polys[tree_polys['Power_tree_size_C3'] > 7]

Salvamos em um arquivo JSON:

In [ ]:
filter_tree.to_file('arvores_segmentadas.json')

## Análise das arvores proximas a linhas e postes:

Importação e visualização das linhas e postes:

In [ ]:
Postes = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/postes.gpkg')

Arvores maiores que 7m a 1m de distancia ou menos de postes:

In [ ]:
Postes = Postes.to_crs(filter_tree.crs)

In [ ]:
Postes = Postes.clip(AOI)

### Arvores e postes a 1m

In [ ]:
postes_prox_tree_1m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=1)

In [ ]:
arvores_prox_postes_1m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_1m.boundary.plot(ax=ax, color='green')
postes_prox_tree_1m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

<Figure size 1500x1500 with 1 Axes>

### Arvores e postes a 3m

In [ ]:
postes_prox_tree_3m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=3)

In [ ]:
arvores_prox_postes_3m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=3)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_3m.boundary.plot(ax=ax, color='green')
postes_prox_tree_3m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

<Figure size 1500x1500 with 1 Axes>

### Arvores e postes a 5m

In [ ]:
postes_prox_tree_5m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=5)

In [ ]:
arvores_prox_postes_5m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=5)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_5m.boundary.plot(ax=ax, color='green')
postes_prox_tree_5m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

<Figure size 1500x1500 with 1 Axes>